In [7]:
# --- Step 1: ライブラリの読み込み ---
import pandas as pd
from sklearn.tree import DecisionTreeClassifier  # 例として決定木モデル
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, KFold, cross_val_score

# --- Step 2: 前処理済みデータの読み込み ---
df = pd.read_csv("train_clean.csv")
df_test = pd.read_csv("test_clean.csv")

#モデル評価
#定義
# ==========================================
# ★追加（エラー防止）: 文字列データを数値に変換 & 欠損値補完
# ※ train_clean.csv に未処理で残っている場合のための安全装置です
# ==========================================
for data in [df, df_test]:
    data['Sex'] = data['Sex'].map({'male': 0, 'female': 1})
    data['Age'] = data['Age'].fillna(df['Age'].median())
# 1. X（説明変数）と y（目的変数）を切り出すセルを先に実行！
# 学習に使う列（前処理済みで数値になっている列）をリストで指定する
features = ['Pclass', 'Sex', 'Age']  # ※使いたい列名に合わせて書き換えてください

X = df[features].values
y = df['Survived'].values

# テストデータ側も同じ列を切り出す
X_test = df_test[features].values

model = DecisionTreeClassifier(random_state=42)

  # データを学習用（50%）とテスト用（50%）にバッサリ半分に分ける
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.5, random_state=42)

# 学習用データ（X_train, y_train）だけでモデルを学習させる
model.fit(X_train, y_train)

# 「学習に使ったデータ」での正解率を表示（カンニングできるので高くなりやすい）
print('正解率(train):{:.3f}'.format(model.score(X_train, y_train)))

# 「まだ見たことがないテストデータ」での正解率を表示（★こちらが本当の実力！）
print('正解率(test):{:.3f}'.format(model.score(X_valid, y_valid)))



# 交差検証して評価
# データを 5 等分（5分割）する設定を作る
kfold = KFold(n_splits=5)

# 5パターンの組み合わせで学習とテストを自動で行い、5回分のスコアを計算する
scores = cross_val_score(model, X, y, cv=kfold)

# 5回それぞれのテストスコアを表示する（例: [0.78, 0.81, 0.75, 0.80, 0.79]）
print('Cross-Validation scores: {}'.format(scores))

# 5回の平均スコアを出す（これがこのモデルの「真の平均実力」！）
import numpy as np
print('Average score: {}'.format(np.mean(scores)))# 交差検証

正解率(train):0.888
正解率(test):0.796
Cross-Validation scores: [0.81564246 0.79775281 0.79775281 0.80898876 0.83146067]
Average score: 0.8103195028560668


# '正解率(train):{:.3f}'.format(数値)
{:.3f}: 「Float（浮動小数点数）を 小数の第 3 位まで表示 してね（4 位で四捨五入）」という意味。

.format(...): {:.3f} の波括弧の中に、計算したスコアの数値を当てはめて埋め込む命令です。

# scores = cross_val_score(model, X, y, cv=kfold)
普通なら「データを分けて、モデルを fit して、score を測る」という作業を 5 回手動で繰り返す必要がありますが、cross_val_score はそれを自動で裏で 5 回分ループ実行してくれます。
cv=kfold と書くことで、「さっき事前に作った『5分割してシャッフルするルール（kfold）』に従って実行してね！」 とやり方を手渡している

# モデル評価のポイント
train と test のスコア比較が超重要！

例えば train が 0.95 で test が 0.60 だった場合、モデルが学習データを暗記しすぎて応用が効かなくなっている 「過学習（Overfitting）」 の状態だと分かります。


In [ ]:
# cv に直接数字の 5 を渡す
scores = cross_val_score(model, X, y, cv=5)

これでも「自動で5分割して交差検証する」という同じような動きになります。

ですが、事前に kfold = KFold(n_splits=5, shuffle=True, random_state=42) と書く方法（cv=kfold）を使う方が、「データを事前にシャッフルする」 などの細かな条件を設定できるため、データ分析ではこちらがよく使われます！

# 💡 過学習を判断する 2 つの現実的ルール
規則①：Train と Validation（CV）の「差（ギャップ）」を見る
最も重要でわかりやすい基準です。

差が 0.02（2%）〜 0.05（5%）以内: 理想的。過学習はほぼ起きておらず、正常。

差が 0.05（5%）〜 0.10（10%）: 黄信号。やや過学習気味（モデルが少し複雑すぎる可能性）。

差が 0.10（10%）以上（例: Train 0.92 / Valid 0.80）: 赤信号（明らかな過学習）。

規則②：数値の絶対値（完璧すぎないか）を見る
Train の正解率が 0.98 〜 1.00（98%〜100%）に到達している:
ノイズや外れ値までも含めて「学習データを丸暗記（暗記学習）」している可能性が非常に高いです。特にデータ数が少ない（タイタニックの 800 件程度など）場合は注意が必要です。

# 🔍 なぜ「交差検証（KFold）」を使うのか？
前半の「ただ半分に分ける方法（ホールドアウト）」だと、たまたま分けたテストデータが易しかったり難しかったりして、スコアが運に左右される 危険があります。

交差検証（KFold）なら、全員が一度ずつテスト役に回るため：

運によるブレがなくなる（平均点が出せる）

全データを学習とテストの両方に有効活用できる

という大きなメリットがあります。Kaggle などでは、この Average score（交差検証の平均値） を見ながら「モデルや前処理をどう改善するか」を判断していくのが標準的な流れです！

In [8]:
#モデルをデータから学習
model.fit(X, y)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [10]:
#木の可視化
from sklearn import tree
import pydotplus
from six import StringIO
from IPython.display import Image

dot_data = StringIO()
single_tree = model.estimators_[0]
tree.export_graphviz(single_tree, out_file=dot_data)
graph = pydotplus.graph_from_dot_data(dot_data.getvalue())
Image(graph.create_png())

ModuleNotFoundError: No module named 'pydotplus'